In [ ]:
# --- ENVIRONMENT SETUP: Environment-Aware Paths ---
import sys, os
from pathlib import Path

try:
    # Standard Colab setup
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    os.system('git clone --depth 1 -q https://github.com/khangnghiem/deep-learning.git /content/deep-learning')
    REPO_ROOT = Path('/content/deep-learning')
except ImportError:
    # Local fallback (assumes running from explorations/ or experiments/)
    cur = Path().resolve()
    REPO_ROOT = cur.parent if cur.name in ('explorations', 'experiments') else cur.parents[1]

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config.paths import GOLD, BRONZE, SILVER, DATA_LAKE, MODELS, PRETRAINED, TRAINED, OPS, MLFLOW_TRACKING_URI, REPOS


In [ ]:
# Cell 1: Setup
from google.colab import drive
drive.mount('/content/drive')
!git clone --depth 1 -q https://github.com/khangnghiem/deep-learning.git /content/deep-learning
import os, sys, glob, json
sys.path.insert(0, '/content/deep-learning')

EXPERIMENT = '014_segformer_polyp'
REPO = 'deep-learning'

print("=" * 60)
print(f"🧪 E2E TEST: {EXPERIMENT}")
print("=" * 60)

In [ ]:
# Cell 2: ✅ Verify data exists
data_path = '/content/drive/MyDrive/data_lake/01_bronze_medical/polypgen'
assert os.path.exists(data_path), f"❌ Data not found: {data_path}"
items = os.listdir(data_path)
assert len(items) > 0, f"❌ Data directory is empty"
print(f"✅ [1/5] Data verified: {len(items)} items")

In [ ]:
# Cell 3: ✅ Verify experiment files
exp_path = f'/content/{REPO}/experiments/{EXPERIMENT}'
required = ['config.yaml', 'train.py', '014_segformer_polyp_l4_pipeline.ipynb', 'README.md']
for f in required:
    assert os.path.exists(os.path.join(exp_path, f)), f"❌ Missing: {f}"
    print(f"  📄 {f}")
print(f"✅ [2/5] All experiment files present")

In [ ]:
# Cell 4: ✅ Verify trained model exists
model_patterns = [
    fstr(TRAINED / '{EXPERIMENT}/**/*'),
    f'/content/{REPO}/experiments/{EXPERIMENT}/**/*.pt',
]
found = []
for p in model_patterns:
    found.extend(glob.glob(p, recursive=True))
assert len(found) > 0, "❌ No trained model found"
for m in found[:5]:
    print(f"  📦 {m}")
print(f"✅ [3/5] Trained model: {len(found)} file(s)")

In [ ]:
# Cell 5: ✅ Verify MLflow run
import mlflow
mlflow.set_tracking_uri('file:///content/drive/MyDrive/mlflow/mlruns')
runs = mlflow.search_runs(experiment_names=[EXPERIMENT], max_results=5)
assert len(runs) > 0, f"❌ No MLflow runs for {EXPERIMENT}"
best = runs.iloc[0]
for col in runs.columns:
    if col.startswith('metrics.'):
        print(f"  📊 {col.replace('metrics.', '')}: {best[col]}")
print(f"✅ [4/5] MLflow run: {best['run_id'][:8]}...")

In [ ]:
# Cell 6: ✅ Verify model inference
import torch
from transformers import SegformerForSemanticSegmentation

# Look for the best model pt file
model_paths = glob.glob(fstr(TRAINED / '{EXPERIMENT}/*.pt'))
assert len(model_paths) > 0, "❌ Cannot load model for inference"

checkpoint = torch.load(model_paths[0], map_location='cpu', weights_only=True)
model = SegformerForSemanticSegmentation.from_pretrained(
    'nvidia/mit-b0',
    num_labels=2,
    ignore_mismatched_sizes=True
)
model.load_state_dict(checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint)
model.eval()

with torch.no_grad():
    dummy = torch.randn(1, 3, 256, 256)
    output = model(dummy)
    logits = output.logits
    print(f"  Output shape: {logits.shape}")
print(f"✅ [5/5] Inference test passed")

In [ ]:
# Cell 7: 🏁 Final summary
print("\n" + "=" * 60)
print("🏁 E2E TEST COMPLETE")
print("=" * 60)
checks = [
    "Data exists and is non-empty",
    "Experiment files are complete",
    "Trained model checkpoint exists",
    "MLflow tracking has logged runs",
    "Model inference produces valid output",
]
for i, c in enumerate(checks, 1):
    print(f"  ✅ {i}. {c}")
print(f"\n🎉 ALL {len(checks)} CHECKS PASSED")
print("=" * 60)

In [ ]:
# Cell 8: Cleanup
from google.colab import runtime
runtime.unassign()